# Model estimation and forecasting

## Define paths, returns, samples, and proxy for volatility

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
DATA_PATH = Path("../data_clean/btc_daily_returns.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df = df.sort_values("Date")
df = df.set_index("Date")

df.head()

,price,log_return
Date,,
2016-01-02,433.437988,-0.206512
2016-01-03,430.010986,-0.793798
2016-01-04,433.091003,0.713712
2016-01-05,431.959991,-0.261490
2016-01-06,429.105011,-0.663130


In [ ]:
print(df.info())

<class 'pandas.DataFrame'>
DatetimeIndex: 3803 entries, 2016-01-02 to 2026-05-31
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   price       3803 non-null   float64
 1   log_return  3803 non-null   float64
dtypes: float64(2)
memory usage: 89.1 KB
None


In [ ]:
returns = df["log_return"].dropna()
returns.name = "BTC daily log return (%)"

returns.head()

Date
2016-01-02   -0.206512
2016-01-03   -0.793798
2016-01-04    0.713712
2016-01-05   -0.261490
2016-01-06   -0.663130
Name: BTC daily log return (%), dtype: float64

In [ ]:
OUT_OF_SAMPLE_FRACTION = 0.20

n_obs = len(returns)
split_idx = int(np.floor((1 - OUT_OF_SAMPLE_FRACTION) * n_obs))

returns_in = returns.iloc[:split_idx].copy()
returns_out = returns.iloc[split_idx:].copy()

print(f"Total number of return observations: {n_obs}")
print(f"In-sample observations:             {len(returns_in)}")
print(f"Out-of-sample observations:         {len(returns_out)}")
print()
print(f"In-sample period:     {returns_in.index.min().date()} to {returns_in.index.max().date()}")
print(f"Out-of-sample period: {returns_out.index.min().date()} to {returns_out.index.max().date()}")

Total number of return observations: 3803
In-sample observations:             3042
Out-of-sample observations:         761

In-sample period:     2016-01-02 to 2024-04-30
Out-of-sample period: 2024-05-01 to 2026-05-31


In [ ]:
# ------------------------------------------------------------
# Construct the out-of-sample realised volatility proxy
# ------------------------------------------------------------

mean_return_in = returns_in.mean()

volatility_proxy_out = (returns_out - mean_return_in) ** 2
volatility_proxy_out.name = "realised_volatility_proxy"

print(f"In-sample mean return: {mean_return_in:.6f}")
print()
print(volatility_proxy_out.head())

In-sample mean return: 0.162355

Date
2024-05-01    17.400140
2024-05-02     1.739983
2024-05-03    36.160564
2024-05-04     2.010098
2024-05-05     0.003136
Name: realised_volatility_proxy, dtype: float64


## Estimate ARCH(1)-normal benchmark model

In [ ]:
# imprt required packages
import arch
from arch import arch_model

In [ ]:
#Specify the simple ARCH(1) model with normal errors
arch1_normal_model = arch_model(
    y=returns_in,
    mean="Constant",
    vol="ARCH",
    p=1,
    dist="normal",
    rescale=False
)

In [ ]:
#Estimate the ARCH(1) normal model
arch1_normal_result = arch1_normal_model.fit(
    disp="off",
    cov_type="classic"
)

print(arch1_normal_result.summary())

                         Constant Mean - ARCH Model Results                         
Dep. Variable:     BTC daily log return (%)   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                             ARCH   Log-Likelihood:               -8255.22
Distribution:                        Normal   AIC:                           16516.4
Method:                  Maximum Likelihood   BIC:                           16534.5
                                              No. Observations:                 3042
Date:                      Sat, Jun 13 2026   Df Residuals:                     3041
Time:                              15:18:22   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu        

In [ ]:
#Extract relevant parameters
arch1_normal_params = pd.DataFrame({
    "estimate": arch1_normal_result.params,
    "std_error": arch1_normal_result.std_err,
    "t_value": arch1_normal_result.tvalues,
    "p_value": arch1_normal_result.pvalues
})

arch1_normal_params

,estimate,std_error,t_value,p_value
mu,0.185263,0.064805,2.858785,4.252668e-03
omega,11.671028,0.363500,32.107330,3.483574e-226
alpha[1],0.164859,0.025733,6.406575,1.488250e-10


In [ ]:
#Extract log-likelihood and information criteria
arch1_normal_info = pd.Series({
    "model": "ARCH(1)",
    "distribution": "Normal",
    "log_likelihood": arch1_normal_result.loglikelihood,
    "aic": arch1_normal_result.aic,
    "bic": arch1_normal_result.bic,
    "n_obs": arch1_normal_result.nobs,
    "num_params": arch1_normal_result.num_params
})

arch1_normal_info

model                  ARCH(1)
distribution            Normal
log_likelihood    -8255.223415
aic                16516.44683
bic               16534.507642
n_obs                     3042
num_params                   3
dtype: object

In [ ]:
#Parameter check
omega_hat = arch1_normal_result.params["omega"]
alpha1_hat = arch1_normal_result.params["alpha[1]"]
unconditional_variance_arch1 = omega_hat / (1 - alpha1_hat)

print(f"Estimated omega:    {omega_hat:.6f}")
print(f"Estimated alpha[1]: {alpha1_hat:.6f}")
print(f"Implied unconditional variance: {unconditional_variance_arch1:.6f}")

Estimated omega:    11.671028
Estimated alpha[1]: 0.164859
Implied unconditional variance: 13.974919


In [ ]:
# Extract and store fitted series
arch1_normal_residuals = arch1_normal_result.resid
arch1_normal_cond_vol = arch1_normal_result.conditional_volatility
arch1_normal_cond_var = arch1_normal_cond_vol ** 2
arch1_normal_std_resid = arch1_normal_residuals / arch1_normal_cond_vol

arch1_normal_fitted = pd.DataFrame({
    "return": returns_in,
    "residual": arch1_normal_residuals,
    "conditional_volatility": arch1_normal_cond_vol,
    "conditional_variance": arch1_normal_cond_var,
    "standardized_residual": arch1_normal_std_resid,
})

arch1_normal_fitted.head()


,return,residual,conditional_volatility,conditional_variance,standardized_residual
Date,,,,,
2016-01-02,-0.206512,-0.391775,3.767500,14.194056,-0.103988
2016-01-03,-0.793798,-0.979061,3.419990,11.696332,-0.286276
2016-01-04,0.713712,0.528448,3.439339,11.829055,0.153648
2016-01-05,-0.261490,-0.446754,3.423020,11.717066,-0.130514
2016-01-06,-0.663130,-0.848393,3.421101,11.703932,-0.247988


In [ ]:
# Inspect standardized residuals
std_resid_summary = arch1_normal_fitted["standardized_residual"].describe()

print(std_resid_summary)
print()
print(f"Mean of standardized residuals:     {arch1_normal_fitted['standardized_residual'].mean():.6f}")
print(f"Variance of standardized residuals: {arch1_normal_fitted['standardized_residual'].var():.6f}")

count    3042.000000
mean       -0.008776
std         1.000126
min       -13.655007
25%        -0.387063
50%        -0.008295
75%         0.408238
max         6.030917
Name: standardized_residual, dtype: float64

Mean of standardized residuals:     -0.008776
Variance of standardized residuals: 1.000252
